# AlexNet on Tiny-ImageNet — FP32, Architecture Variants & QAT (INT8)

**Author:** Rafael
**Dataset:** Tiny-ImageNet-200 (200 classes, 64×64 RGB)
**Goal:** Train and compare three AlexNet variants in FP32, then apply
Quantization-Aware Training (QAT) to obtain INT8 versions. Report **Top-1**
and **Top-5** accuracy for every model.

## Notebook layout

1. Configuration & reproducibility
2. Dataset & loaders
3. Model definitions (`AlexNet`, `AlexNet3x3`, `AlexNetSmallKernel`)
4. QAT helpers (fuse + prepare)
5. FP32 training (3 models)
6. QAT training (3 models) and INT8 conversion
7. Evaluation — Top-1 and Top-5 — for FP32 and INT8
8. Final comparison table & ranking


## 1. Configuration & reproducibility

Everything that controls the experiment lives in a single `ExperimentConfig`
dict. The seed is fixed and `set_seed` propagates it to `random`, `numpy`,
and `torch` (CPU & CUDA).


In [1]:
# Standard library
import os
from pathlib import Path

# Third-party
import torch
import torch.nn as nn
import torch.optim as optim
import torch.ao.quantization as tq
from torch.utils.data import DataLoader
from torchvision.models import alexnet

# Local
from ml_utils import (
    get_system_info,
    evaluate_topk,
    train_and_evaluate,
    register_model,
    MODEL_REGISTRY,
    build_comparison_table,
    create_imagenet_loaders,
    load_model,
    count_parameters,
    create_results_summary,
    setup_experiment,
    build_default_config, 
    get_system_info,
    convert_to_int8,
    build_qat,
    load_best_model,
    disk_mb,
    run_fp32_training,
    run_qat_training
)

# Quantization backend — must be set before any QAT prep / convert
torch.backends.quantized.engine = "fbgemm"


/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
device = setup_experiment(SEED, cudnn_benchmark=True)

config = build_default_config(
    seed=SEED,
    device=device,
    save_dir="./models/alexnet_qat"
)

NUM_CLASSES = config["num_classes"]

print(get_system_info())

{'device': 'cuda', 'torch_version': '2.5.1+cu121', 'cuda_available': True, 'cuda_version': '12.1', 'gpu_count': 1, 'gpu_name': 'NVIDIA GeForce RTX 4060 Laptop GPU', 'gpu_memory_gb': 8.186822656}


## 2. Dataset & loaders

We use Tiny-ImageNet-200 from Kaggle. The train folder is split 90/10 into
train/val with a **seeded generator** so the split is identical on every run.

* Training transforms: light augmentation (random crop + hflip + color jitter)
* Validation transforms: deterministic resize + center crop

Note that we build two `ImageFolder` objects (one per transform) and index
them with the same `Subset` indices — this keeps train-time augmentation
separate from val-time deterministic preprocessing.


In [3]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
print("Tiny-ImageNet train path:", train_path)


Tiny-ImageNet train path: /home/rafael/.cache/kagglehub/datasets/akash2sharma/tiny-imagenet/versions/1/tiny-imagenet-200/train


In [4]:
train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(
    dataset_path=train_path,
    img_size=config["img_size"],
    batch_size=config["batch_size"],
    num_workers=config["num_workers"],
    train_split=config["train_val_split"],
    seed=config["seed"],
    pin_memory=config["pin_memory"],
    persistent_workers=True,
)

NUM_CLASSES = config["num_classes"]

print(f"Train samples: {len(train_ds):,}")
print(f"Val   samples: {len(val_ds):,}")
print(f"Classes:       {NUM_CLASSES}")
print(f"Batches/epoch: train={len(train_loader)}, val={len(val_loader)}")

Train samples: 90,000
Val   samples: 10,000
Classes:       200
Batches/epoch: train=1406, val=157


## 3. Model architectures

Three variants of AlexNet are compared:

| Name | Backbone | Notes |
|---|---|---|
| `AlexNet_FP32` | torchvision `alexnet(IMAGENET1K_V1)` | Last FC replaced by `Linear(4096, 200)`. Pretrained, fine-tuned. |
| `AlexNet3x3` | All 3×3 conv stack, AdaptiveAvgPool(6×6) → 3-layer MLP | From-scratch, AlexNet-style head. |
| `AlexNetSmallKernel` | All 3×3 convs, AdaptiveAvgPool(1×1) → single `Linear(256, 200)` | Lightweight, ~250× smaller classifier. |


In [5]:
class AlexNetTV(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        base = alexnet(weights="IMAGENET1K_V1")
        base.classifier[6] = nn.Linear(4096, num_classes)
        for name, module in base.features.named_children():
            if isinstance(module, nn.ReLU):
                setattr(base.features, name, nn.ReLU(inplace=False))

        self.quant = tq.QuantStub()
        self.features = base.features
        self.avgpool = base.avgpool
        self.classifier = base.classifier
        self.dequant = tq.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        x = self.dequant(x)
        return x


def build_alexnet(num_classes: int = NUM_CLASSES) -> nn.Module:
    return AlexNetTV(num_classes)


class AlexNet3x3(nn.Module):
    """All-3x3 AlexNet-like CNN modified for QAT/INT8 compatibility."""

    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.quant = tq.QuantStub()
        self.dequant = tq.DeQuantStub()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=2, padding=1),   # 0
            nn.ReLU(inplace=False),                      # 1
            nn.MaxPool2d(2),                             # 2
            nn.Conv2d(64, 192, 3, padding=1),            # 3
            nn.ReLU(inplace=False),                      # 4
            nn.MaxPool2d(2),                             # 5
            nn.Conv2d(192, 384, 3, padding=1),           # 6
            nn.ReLU(inplace=False),                      # 7
            nn.Conv2d(384, 256, 3, padding=1),           # 8
            nn.ReLU(inplace=False),                      # 9
            nn.Conv2d(256, 256, 3, padding=1),           # 10
            nn.ReLU(inplace=False),                      # 11
            nn.AdaptiveAvgPool2d((6, 6)),                # 12
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=False),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=False),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.quant(x)
        x = self.features(x)
        x = self.classifier(x)
        x = self.dequant(x)
        return x


class AlexNetSmallKernel(nn.Module):
    """All-3x3 CNN with global-pool, modified for QAT/INT8 compatibility."""

    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.quant = tq.QuantStub()
        self.dequant = tq.DeQuantStub()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1),    # 0
            nn.ReLU(inplace=False),                       # 1
            nn.MaxPool2d(2),                              # 2
            nn.Conv2d(64, 128, 3, padding=1),             # 3
            nn.ReLU(inplace=False),                       # 4
            nn.MaxPool2d(2),                              # 5
            nn.Conv2d(128, 256, 3, padding=1),            # 6
            nn.ReLU(inplace=False),                       # 7
            nn.Conv2d(256, 256, 3, padding=1),            # 8
            nn.ReLU(inplace=False),                       # 9
            nn.Conv2d(256, 256, 3, padding=1),            # 10
            nn.ReLU(inplace=False),                       # 11
            nn.AdaptiveAvgPool2d((1, 1)),                 # 12
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.quant(x)
        x = self.features(x)
        x = self.classifier(x)
        x = self.dequant(x)
        return x


# Smoke test — verify forward pass shapes
def _sanity_check():
    x = torch.randn(2, 3, config["img_size"], config["img_size"])
    for ctor in (build_alexnet, AlexNet3x3, AlexNetSmallKernel):
        m = ctor()
        m.eval()
        with torch.no_grad():
            y = m(x)
        assert y.shape == (2, NUM_CLASSES), (ctor, y.shape)
        print(f"  {ctor.__name__:25s} OK -> {tuple(y.shape)}, "
              f"params={count_parameters(m)/1e6:.2f}M")

_sanity_check()

  build_alexnet             OK -> (2, 200), params=57.82M
  AlexNet3x3                OK -> (2, 200), params=57.61M
  AlexNetSmallKernel        OK -> (2, 200), params=1.60M


## 5. QAT helpers 

In the original notebook the QAT prep was duplicated three times. Here it
is one function that:

1. puts the model in `train()` mode (required by `prepare_qat`),
2. sets the fbgemm `qconfig`,
3. fuses a user-supplied list of `[conv, relu]` patterns,
4. returns `prepare_qat(model)`.

Only Conv+ReLU (or Conv+BN+ReLU) is fused — never anything involving
`MaxPool` or `AdaptiveAvgPool`, which is invalid and crashes `fuse_modules`.


In [6]:
# Fuse maps for each architecture — read off the layer-index comments above.
# Pattern: [conv_idx, relu_idx] (a MaxPool between two blocks is NOT fused).
FUSE_MAP_ALEXNET_TV = [          # torchvision AlexNet.features layout
    ["0", "1"], ["3", "4"], ["6", "7"], ["8", "9"], ["10", "11"]
]
FUSE_MAP_3X3 = [
    ["0", "1"], ["3", "4"], ["6", "7"], ["8", "9"], ["10", "11"]
]
FUSE_MAP_SMALL = [
    ["0", "1"], ["3", "4"], ["6", "7"], ["8", "9"], ["10", "11"]
]

register_model(
    "alexnet_fp32",
    build_alexnet,
    fuse_map=FUSE_MAP_ALEXNET_TV,
    fuse_root_attr="features",
    lr=config["lr_pretrained"],
)

register_model(
    "alexnet_3x3",
    AlexNet3x3,
    fuse_map=FUSE_MAP_3X3,
    fuse_root_attr="features",
    lr=config["lr_from_scratch"],
)

register_model(
    "alexnet_small_kernel",
    AlexNetSmallKernel,
    fuse_map=FUSE_MAP_SMALL,
    fuse_root_attr="features",
    lr=config["lr_from_scratch"],
)

## 7. FP32 training — three architectures

In [7]:
criterion = nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"])

fp32_models, fp32_training_results = run_fp32_training(
    MODEL_REGISTRY, train_loader, val_loader, criterion, config, device
)

Training: alexnet_fp32 (lr=0.0001, epochs=2)


Evaluation: 100%|██████████| 157/157 [00:01<00:00, 114.80it/s, acc=23.20%]


Epoch   1/2 | Train Loss: 4.4081 | Train Acc: 12.27% | Val Loss: 3.7901 | Val Acc: 23.20% | Time: 48.2s


Evaluation: 100%|██████████| 157/157 [00:01<00:00, 112.99it/s, acc=28.43%]


Epoch   2/2 | Train Loss: 3.9686 | Train Acc: 19.82% | Val Loss: 3.5696 | Val Acc: 28.43% | Time: 52.1s


...

Evaluation: 100%|██████████| 157/157 [00:01<00:00, 98.30it/s, acc=3.32%] 


Epoch   1/2 | Train Loss: 5.1763 | Train Acc: 1.62% | Val Loss: 4.9662 | Val Acc: 3.32% | Time: 22.8s


Evaluation: 100%|██████████| 157/157 [00:01<00:00, 106.91it/s, acc=8.56%]
/home/rafael/Documents/alexnet_rafael/ml_utils.py:254: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


Epoch   2/2 | Train Loss: 4.8129 | Train Acc: 5.39% | Val Loss: 4.5555 | Val Acc: 8.56% | Time: 23.6s
[best] loss=4.5555  top1=8.56%  top5=25.72%


best_val_acc,▁█
epoch_time,▁█
final_loss,▁
final_top1,▁
final_top5,▁
train_acc,▁█
train_loss,█▁
val_acc,▁█
val_loss,█▁
best_val_acc,8.56
epoch_time,23.57999


## 8. Quantization-Aware Training (QAT)

We fine-tune each FP32 checkpoint for a few epochs with fake-quant observers
inserted, then `convert` to true INT8 on CPU.

> **Note on the engine.** We set `torch.backends.quantized.engine = "fbgemm"`
> at the top of the notebook. QAT *training* happens on GPU; `convert` and
> INT8 *inference* must run on CPU.


In [8]:
criterion = nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"])

qat_models, qat_training_results = run_qat_training(
    MODEL_REGISTRY, train_loader, val_loader, criterion, config, device
)

/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


Training: qat_alexnet_fp32 (lr=1e-05, epochs=2)


Evaluation: 100%|██████████| 157/157 [00:01<00:00, 109.92it/s, acc=29.29%]


Epoch   1/2 | Train Loss: 3.8443 | Train Acc: 22.28% | Val Loss: 3.5244 | Val Acc: 29.29% | Time: 51.1s


Evaluation: 100%|██████████| 157/157 [00:01<00:00, 107.57it/s, acc=30.08%]


...

Evaluation: 100%|██████████| 157/157 [00:01<00:00, 83.54it/s, acc=8.90%]


Epoch   1/2 | Train Loss: 4.6343 | Train Acc: 8.07% | Val Loss: 4.5209 | Val Acc: 8.90% | Time: 44.6s


Evaluation: 100%|██████████| 157/157 [00:01<00:00, 83.62it/s, acc=9.09%]


Epoch   2/2 | Train Loss: 4.6154 | Train Acc: 8.48% | Val Loss: 4.5185 | Val Acc: 9.09% | Time: 44.0s
[best] loss=4.5188  top1=8.87%  top5=27.49%


best_val_acc,▁█
epoch_time,█▁
final_loss,▁
final_top1,▁
final_top5,▁
train_acc,▁█
train_loss,█▁
val_acc,▁█
val_loss,█▁
best_val_acc,9.09
epoch_time,44.01898


## 9. INT8 conversion and CPU evaluation

`tq.convert` materialises true quantised ops; this must run on CPU and the
model must be in `eval()` mode. Because INT8 inference is CPU-only, we
build a small CPU-side val loader (`num_workers=0` is safer here — fewer
chances of hangs after a GPU run) and reuse `evaluate_topk`.


In [9]:
val_loader_cpu = DataLoader(
    val_ds,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

cpu_device = torch.device("cpu")

int8_models = {}

for name in ["alexnet_fp32", "alexnet_3x3", "alexnet_small_kernel"]:
    model = qat_models[name]

    # ensure clean CPU state before conversion
    model = model.cpu().eval()

    # convert QAT model to INT8 (CPU-only)
    int8_models[name] = convert_to_int8(model)

# Save INT8 state dicts
for name, m in int8_models.items():
    torch.save(m.state_dict(), os.path.join(config["save_dir"], f"{name}.pth"))

print("INT8 conversion done.")

int8_metrics = {}

for name, m in int8_models.items():
    m = m.cpu().eval()

    for n, p in m.named_parameters():
        if p.device.type != "cpu":
            print("PARAM on", p.device, n)
    for n, b in m.named_buffers():
        if b.device.type != "cpu":
            print("BUFFER on", b.device, n)

    metrics = evaluate_topk(
        m,
        val_loader_cpu,
        criterion,
        cpu_device
    )

    int8_metrics[name] = metrics

    print(
        f"{name:22s} | loss={metrics['loss']:.4f} | "
        f"top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%"
    )

INT8 conversion done.
alexnet_fp32           | loss=3.4900 | top1=30.06% | top5=56.63%
alexnet_3x3            | loss=4.4834 | top1=9.57% | top5=27.53%
alexnet_small_kernel   | loss=4.5182 | top1=8.95% | top5=27.19%


## 10. FP32 evaluation — reload best checkpoints

Reload each best FP32 checkpoint from disk to make sure we're evaluating
the *best* epoch (not whatever was in memory at the end of training).


In [10]:
fp32_models = {
    "AlexNet_FP32":      load_best_model("alexnet_fp32",         build_alexnet,        config["save_dir"], device),
    "AlexNet3x3_FP32":   load_best_model("alexnet_3x3",          AlexNet3x3,           config["save_dir"], device),
    "AlexNetSmall_FP32": load_best_model("alexnet_small_kernel", AlexNetSmallKernel,   config["save_dir"], device),
}

fp32_metrics = {}
for name, m in fp32_models.items():
    metrics = evaluate_topk(m, val_loader, criterion, device)
    fp32_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | "
          f"top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")


/home/rafael/Documents/alexnet_rafael/ml_utils.py:254: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location=device)


AlexNet_FP32           | loss=3.5696 | top1=28.43% | top5=55.01%
AlexNet3x3_FP32        | loss=4.5783 | top1=8.25% | top5=24.38%
AlexNetSmall_FP32      | loss=4.5555 | top1=8.56% | top5=25.72%


## 11. Final comparison — Top-1, Top-5, size, params

This is the single artefact to read at the end:

* **Top-1 / Top-5 accuracy** on the validation split
* **Trainable parameters** (millions)
* **Disk size** of the checkpoint (MB) — for INT8 this is the *real* on-disk size


In [11]:
rows = []

# FP32 rows
fp32_files = {
    "AlexNet_FP32":      "alexnet_fp32_best.pth",
    "AlexNet3x3_FP32":   "alexnet_3x3_best.pth",
    "AlexNetSmall_FP32": "alexnet_small_kernel_best.pth",
}
for name, m in fp32_models.items():
    rows.append({
        "model":     name,
        "precision": "FP32",
        "top1_%":    fp32_metrics[name]["top1"],
        "top5_%":    fp32_metrics[name]["top5"],
        "loss":      fp32_metrics[name]["loss"],
        "params_M":  count_parameters(m) / 1e6,
        "size_MB":   disk_mb(os.path.join(config["save_dir"], fp32_files[name])),
    })

int8_name_map = {
    "alexnet_fp32": "AlexNet_FP32",
    "alexnet_3x3": "AlexNet3x3_FP32",
    "alexnet_small_kernel": "AlexNetSmall_FP32",
}

int8_files = {
    "alexnet_fp32": "alexnet_fp32.pth",
    "alexnet_3x3": "alexnet_3x3.pth",
    "alexnet_small_kernel": "alexnet_small_kernel.pth",
}

for name, m in int8_models.items():
    rows.append({
        "model": f"{name}_INT8",
        "precision": "INT8",
        "top1_%": int8_metrics[name]["top1"],
        "top5_%": int8_metrics[name]["top5"],
        "loss": int8_metrics[name]["loss"],
        "params_M": count_parameters(fp32_models[int8_name_map[name]]) / 1e6,
        "size_MB": disk_mb(
            os.path.join(config["save_dir"], int8_files[name])
        ),
    })

df = build_comparison_table(rows)
df.to_csv(os.path.join(config["save_dir"], "final_comparison.csv"), index=False)
df


,model,precision,top1_%,top5_%,loss,params_M,size_MB
0,AlexNet_FP32,FP32,28.43,55.01,3.569552,57.823240,220.584150
1,AlexNetSmall_FP32,FP32,8.56,25.72,4.555502,1.602376,6.117510
2,AlexNet3x3_FP32,FP32,8.25,24.38,4.578262,57.605128,219.752039
3,alexnet_fp32_INT8,INT8,30.06,56.63,3.490038,57.823240,55.332716
4,alexnet_3x3_INT8,INT8,9.57,27.53,4.483409,57.605128,55.124596
5,alexnet_small_kernel_INT8,INT8,8.95,27.19,4.518156,1.602376,1.561041


In [12]:
print("=" * 72)
print("RANKING BY TOP-1 ACCURACY (all precisions)")
print("=" * 72)
ranked = df.sort_values("top1_%", ascending=False).reset_index(drop=True)
for i, row in ranked.iterrows():
    print(f"{i+1:2d}. {row['model']:22s} [{row['precision']}] | "
          f"top1={row['top1_%']:6.2f}% | top5={row['top5_%']:6.2f}% | "
          f"params={row['params_M']:6.2f}M | size={row['size_MB']:6.2f}MB")


RANKING BY TOP-1 ACCURACY (all precisions)
 1. alexnet_fp32_INT8      [INT8] | top1= 30.06% | top5= 56.63% | params= 57.82M | size= 55.33MB
 2. AlexNet_FP32           [FP32] | top1= 28.43% | top5= 55.01% | params= 57.82M | size=220.58MB
 3. alexnet_3x3_INT8       [INT8] | top1=  9.57% | top5= 27.53% | params= 57.61M | size= 55.12MB
 4. alexnet_small_kernel_INT8 [INT8] | top1=  8.95% | top5= 27.19% | params=  1.60M | size=  1.56MB
 5. AlexNetSmall_FP32      [FP32] | top1=  8.56% | top5= 25.72% | params=  1.60M | size=  6.12MB
 6. AlexNet3x3_FP32        [FP32] | top1=  8.25% | top5= 24.38% | params= 57.61M | size=219.75MB


## 12. Persist experiment summary

A single JSON next to the checkpoints captures: config, system info,
per-model metrics. Re-running this notebook with the same seed should
reproduce these numbers up to non-determinism in cuDNN kernels (we set
`benchmark=True` for speed; flip to `False` for exact bit reproducibility).


In [13]:
create_results_summary(
    results={
        "fp32_metrics": fp32_metrics,
        "int8_metrics": int8_metrics,
        "fp32_training_results": fp32_training_results,
        "qat_training_results": qat_training_results,
    },
    config=config.to_dict(),
    output_path=os.path.join(
        config["save_dir"],
        "experiment_summary.json",
    ),
)